# JARVIS-MKIII Fine-Tuning Notebook
Train a custom JARVIS LLM using Unsloth + LoRA on Llama-3.1-8B-Instruct

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install transformers datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## Upload and Load Dataset
Upload your `dataset.jsonl` file when prompted.

In [ ]:
from google.colab import files
import json

print("Upload your dataset.jsonl file:")
uploaded = files.upload()

dataset_path = "dataset.jsonl"
with open(dataset_path, "w") as f:
    for fname, data in uploaded.items():
        f.write(data.decode("utf-8"))

# Load entries
entries = []
with open(dataset_path) as f:
    for line in f:
        line = line.strip()
        if line:
            entries.append(json.loads(line))

print(f"Loaded {len(entries)} training entries")

In [ ]:
from datasets import Dataset

JARVIS_SYSTEM = """You are JARVIS, an advanced AI assistant created by Khalid. 
You speak in a dry, clipped British tone. You always address the user as 'sir'.
You never say 'certainly', 'of course', 'absolutely', 'sure', or 'great'.
You keep responses concise — maximum 2 sentences for simple queries."""

def format_prompt(entry):
    messages = [
        {"role": "system", "content": JARVIS_SYSTEM},
        {"role": "user", "content": entry["instruction"]},
        {"role": "assistant", "content": entry["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

raw_dataset = Dataset.from_list(entries)
dataset = raw_dataset.map(format_prompt)
print(f"Dataset formatted. Sample:\n{dataset[0]['text'][:300]}")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "/content/jarvis_checkpoints",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Training complete. Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
model.save_pretrained("/content/jarvis_lora")
tokenizer.save_pretrained("/content/jarvis_lora")
print("LoRA adapter saved to /content/jarvis_lora")

In [ ]:
print("Merging and exporting to GGUF (Q4_K_M)...")
model.save_pretrained_gguf(
    "/content/jarvis_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("GGUF export complete!")
print("\nFiles in /content/jarvis_gguf:")
import os
for f in os.listdir("/content/jarvis_gguf"):
    size = os.path.getsize(f"/content/jarvis_gguf/{f}") / (1024**3)
    print(f"  {f}: {size:.2f} GB")

In [ ]:
from google.colab import files
import zipfile, os

print("Zipping GGUF model for download...")
with zipfile.ZipFile("/content/jarvis_mkiii_q4km.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir("/content/jarvis_gguf"):
        zf.write(f"/content/jarvis_gguf/{fname}", fname)

files.download("/content/jarvis_mkiii_q4km.zip")
print("Download initiated.")

## Next Steps
After downloading:
1. Extract the .gguf file
2. Create a Modelfile (see training/README.md)
3. Run: `ollama create jarvis-mkiii -f Modelfile`
4. Update config/settings.py: set `LLM_MODEL = "jarvis-mkiii"`